# artgen from Python

The `artgen` package is a thin client: each call runs the artgen CLI (`node tools/artgen/artgen.js … --json`) in a game repo and hands back dicts and Pillow images. This notebook works on a scratch copy of the `iso-dungeon` fixture so nothing in the repo changes.

Needs Node 20+ on `PATH` and an `artgen.js`: the fixture's committed install (`npm run fixtures:install` restores the bundle after a fresh clone) or `ARTGEN_CLI`.

In [ ]:
import os, shutil, tempfile
from pathlib import Path
from artgen import Artgen

try:
    from IPython.display import display
except ImportError:  # plain Python
    display = print

fixture = Path("../../../examples/iso-dungeon").resolve()
project = Path(tempfile.mkdtemp(prefix="artgen-nb-")) / "iso-dungeon"
shutil.copytree(fixture / "art", project / "art")

ag = Artgen(project, cli=os.environ.get("ARTGEN_CLI") or fixture / "tools" / "artgen" / "artgen.js")
print(ag.version(), "—", ag.direction()["locked"]["id"])

## A texture

A material recipe under the project's locked direction: the tile, its normal map and a 3×3 repeat preview, plus the seam and repetition metrics the gate uses.

In [ ]:
tex = ag.texture("stone", size=32, seed=7)
print("seam ratio", tex["seam"]["ratio"], "| repetition issues:", tex["issues"] or "none")
display(tex.preview)

Bulk work stays in Python: render several materials and lay them out with Pillow.

In [ ]:
from PIL import Image

names = ["stone", "wood", "brick", "grass"]
tiles = [ag.texture(m, size=32, seed=3).tile for m in names]
board = Image.new("RGBA", (len(tiles) * 34 - 2, 32))
for i, t in enumerate(tiles):
    board.paste(t, (i * 34, 0))
board = board.resize((board.width * 4, board.height * 4), Image.NEAREST)
display(board)

For bulk work, a **session** keeps one MCP server process open: the engine starts once, and images come back inline instead of as files (several times faster than a CLI call per item).

In [ ]:
with ag.session() as s:
    previews = [s.call("texture", material=m, size=32, seed=5).images[0] for m in names]
print(len(previews), "3×3 previews from one session:", [im.size for im in previews])

## A sheet

The review sheet of a finished asset (the same sheet the art-reviewer scores), and its render strip.

In [ ]:
ids = [s["id"] for s in ag.status()]
print(ids)
sheet = ag.review("skeleton-knight")
print(sheet["sheet"], "~", sheet["tokens"], "image tokens; gate", "pass" if sheet["pass"] else "FAIL")
display(sheet.sheet)
render = ag.render("skeleton-knight")
display(render.scaled)

## An effect

A particle preset: the frame strip and an onion-skin sheet (a GIF and an APNG are written beside them).

In [ ]:
fx = ag.fx("sparks", size=24, frames=6, seed=2)
print(fx.gif.name, fx["fill"])
display(fx.onion)